In [1]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

import os

# Created following the tutorial: https://www.youtube.com/watch?v=CtzfbUwrYGI

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

In [4]:
train_data = torchvision.datasets.CIFAR10(root = './data', train = True, transform = transform, download = True)
test_data = torchvision.datasets.CIFAR10(root = './data', train = False, transform = transform, download = True)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle = True, num_workers = 2)
test_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle = True, num_workers = 2)

In [5]:
class_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [6]:
class NeuralNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 12, 5) 
        self.pool = nn.MaxPool2d(2,2) 
        self.conv2 = nn.Conv2d(12,24,5)
        self.fc1 = nn.Linear(24*5*5, 120)
        self.fc2 = nn.Linear(120,84)
        self.fc3 = nn.Linear(84,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x,1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
        

In [7]:
net = NeuralNet()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr = 0.001, momentum = 0.9)

# Training with training data

In [8]:
for epoch in range(50):
    print(f'Training epoch {epoch}')

    running_loss = 0.0

    for i, data in enumerate(train_loader):
        inputs, labels = data

        optimizer.zero_grad()

        outputs = net(inputs)

        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss =+ loss.item()

    print(f'Loss: {running_loss / len(train_loader):.4f}')

Training epoch 0
Loss: 0.0012
Training epoch 1
Loss: 0.0008
Training epoch 2
Loss: 0.0011
Training epoch 3
Loss: 0.0010
Training epoch 4
Loss: 0.0006
Training epoch 5
Loss: 0.0010
Training epoch 6
Loss: 0.0005
Training epoch 7
Loss: 0.0009
Training epoch 8
Loss: 0.0007
Training epoch 9
Loss: 0.0006
Training epoch 10
Loss: 0.0008
Training epoch 11
Loss: 0.0005
Training epoch 12
Loss: 0.0005
Training epoch 13
Loss: 0.0008
Training epoch 14
Loss: 0.0008
Training epoch 15
Loss: 0.0004
Training epoch 16
Loss: 0.0006
Training epoch 17
Loss: 0.0005
Training epoch 18
Loss: 0.0006
Training epoch 19
Loss: 0.0001
Training epoch 20
Loss: 0.0008
Training epoch 21
Loss: 0.0004
Training epoch 22
Loss: 0.0005
Training epoch 23
Loss: 0.0004
Training epoch 24
Loss: 0.0003
Training epoch 25
Loss: 0.0003
Training epoch 26
Loss: 0.0005
Training epoch 27
Loss: 0.0003
Training epoch 28
Loss: 0.0002
Training epoch 29
Loss: 0.0001
Training epoch 30
Loss: 0.0003
Training epoch 31
Loss: 0.0005
Training epoch 32


In [9]:
torch.save(net.state_dict(), "trained_net.pth")

In [10]:
net = NeuralNet()
net.load_state_dict(torch.load("trained_net.pth"))

<All keys matched successfully>

# Testing with test data 

In [11]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct/total

print(f'Accuracy: {accuracy}%')        

Accuracy: 94.918%


In [12]:
new_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

def load_image(image_path):
    image = Image.open(image_path)
    image = new_transform(image)
    image = image.unsqueeze(0)
    return image



# Creating list of paths to manually collected images

In [13]:
images_path = ["./test_images/" + x for x in os.listdir("./test_images")]

In [14]:
images = [load_image(img) for img in images_path]

In [15]:
net.eval()
with torch.no_grad():
    for count, image in enumerate(images):
        path = images_path[count].split("/")[-1]
        aminal = path.split(" ")[1]
        print(f'Label {count}: {aminal}')
        output = net(image)
        _, predicted = torch.max(output, 1)
        print(f'Prediction {count}: {class_names[predicted.item()]}')
        print("------------------")

Label 0: dog
Prediction 0: bird
------------------
Label 1: frog
Prediction 1: frog
------------------
Label 2: plane
Prediction 2: plane
------------------
Label 3: dog
Prediction 3: dog
------------------
Label 4: truck
Prediction 4: car
------------------
Label 5: dog
Prediction 5: dog
------------------
Label 6: cat
Prediction 6: cat
------------------
Label 7: airplane
Prediction 7: plane
------------------
Label 8: deer
Prediction 8: deer
------------------
Label 9: cast
Prediction 9: bird
------------------
Label 10: frog
Prediction 10: frog
------------------
Label 11: cat
Prediction 11: bird
------------------
Label 12: horse
Prediction 12: deer
------------------
Label 13: ship
Prediction 13: ship
------------------
Label 14: frog
Prediction 14: frog
------------------
Label 15: truck
Prediction 15: car
------------------
Label 16: plane
Prediction 16: plane
------------------
Label 17: ship
Prediction 17: ship
------------------
